## 009.ipynb

In [1]:
# 감성 분석 (Sentiment Analysis)
# 텍스트에서 감정, 의견, 태도를 자동으로 파악하는 NLP기술
# 1. 사전 기반 (Lexicon-based) 방식
# => 미리 각 단어에 점수를 매겨둔 사전이 있고, 
#    테스트의 단어들을 그 사전에 찾아서 점수를 합산 (머신러닝 없이도 동작 가능

In [2]:
# TextBlob => 단어 평균 점수 + 부정어/강조어
# AFINN => -5 ~ +5 점수, 빠름
# VANDER => SNS/구두점/이모티콘 특화

In [3]:
# TextBlob
# - polarity(극성) => 얼마나 긍정/부정인가 
# -1(완전 부정) ~ 0(중립) ~ 1(완전 긍정)
# - subjectivity(주관성) => 얼마나 주관적인가
# 0.0(완전 객관적 사실) ~ 1.0(완전 주관적 감정/의견)

# 두개를 같이 비교하는 이유
# - 긍정처럼 보이지만 사실(개관적) -> 진짜 감상 X
# - 긍정 + 주관적 -> 진짜 긍정 리뷰

In [4]:
# textblob_predict = np.array(textblob_predict) > 0.1
# => 0.0(중립)을 neg/pos로 구분하기 어려움
# 0.1은 긍정을 최대한 보수적으로 보는 기준(애매하게 긍정인건 믿지 말자)
# 0.1 => 임계값(Threshold)

# 어떤 값이 임계값으로 최선인지는 알 수 없음 (직접 조절 필요)
# => 하이퍼파라미터(Hyperparameter) : 직접 조절해야 하는 값

In [5]:
# TextBlob 단계별 동작
# 1. 사전(lexicon)에 단어별 점수 저장
# 2. 텍스트를 토큰(단어)으로 쪼갬
# 3. 부정어/강조어 처리
# 4. 점수 평균 계산

In [6]:
import re
from konlpy.tag import Okt

class CustomTextBlob:
    def __init__(self, language='en'):
        self.language = language
        self.okt = Okt() if language == 'ko' else None

        # 감정 사전(Lexicon)
        self.lexicon = {
            'great' : 0.8, 'good' : 0.5, 'happy' : 0.7,
            'bad' : -0.6, 'terrible' : -0.9, 'sad' : -0.5,
            '좋다' : 0.6, '최고' : 0.9, '행복하다' : 0.8,
            '나쁘다' : -0.6, '최악' : -0.9, '슬프다' : -0.7
        }
        # 실제 TextBlob은 이런 단어가 수만 개 들어있음
        # 사전에 없는 단어는 무시 => 점수 기여 없음
       
        # 부정어 / 강조어 처리
        # 부정어 : 감정 단어 점수를 뒤집음
        self.negation = {'not', 'never', 'no', '안', '못', '아니'}
        # 강조어 : 감정 단어 점수를 배율로 키움
        self.intensifiers = {'very' : 1.5, '매우' : 1.5}

    # 토큰화(Tokenization)
    def tokenize(self, text):
        # 한국어 형태소로 나눠서 단어만 추출
        if self.language == 'ko':
            return [word for word, pos in self.okt.pos(text, stem=True)]

        # 영어 : 소문자로 바꾸고 단어만 추출
        else:
            return re.findall(r'\w+', text.lower())
            # \w : 문자 + 숫자 + _(unerscore)
            # + : 1개 이상 반복

    # 점수 계산
    def analyze(self, text):
        tokens = self.tokenize(text)
        scores = []
        negate = False  # 부정어 스위치
        intensity = 1.0 # 강조어 배율 (초기값)


        for token in tokens:

            # 경우 1 : 강조어 발견
            if token in self.intensifiers:
                intensity = self.intensifiers[token]
                continue    # 강조어 자체는 점수 없음, 배율만 저장하고 다음 단어로 점프

            if token in self.negation:
                negate = True   # 스위치 ON
                continue        # 부정어 자체 점수 없음, 다음으로

            # 경우 3 : 감정 단어 발견
            if token in self.lexicon:
                score = self.lexicon[token] * intensity # 강조어 적용

                if negate:
                    score *= -0.5 # 부정어 적용
                    negate = False # 한 번 쓰면 스위치 OFF

                scores.append(score)
                intensity = 1.0     # 강조어도 초기화
    
        # 점수 산출
        if not scores:
            return 0.0
    
        return sum(scores) / len(scores)  # 점수 평균
            


In [7]:
blob = CustomTextBlob()
print(blob.analyze('this movie is good'))       
print(blob.analyze('this movie is very good'))  
print(blob.analyze('this movie is not good'))   
print(blob.analyze('not very good'))   

0.5
0.75
-0.25
-0.375


## 010.ipynb

In [8]:
# Word2Vec 
# 사람이 점수를 정하지 않음, 텍스트에서 스스로 학습
# 같은 맥락에서 자주 등장하는 단어는 의미가 비슷하다

In [ ]:
import re
from gensim.models import Word2Vec

corpus = [
    'i love nlp',
    'i love machine learning',
    'nlp is fun',
    'machine learning is powerful',
    'i enjoy deep learning',
    'natural language processing is interesting'
]

sentences = [re.findall(r'\w+', text.lower()) for text in corpus]

model = Word2Vec(
    sentences = sentences,  # 단어 리스트의 리스트
    vector_size = 100,      # 각 단어를 100차원 벡터로 표현
    window=3,               # 주변 몇 개 단어까지 볼지
    min_count=1,            # 최소 등장 횟수 (이하면 무시)
    # 데이터가 작으니까 1번만, 대용량 데이터의 경우 5 이상으로 설정
    sg = 1,                 # 0=CBOW, 1=Skip-gram
)

# CBOW (sg=0)       : 주변 단어 -> 중심 단어 예측
# Skip-gram (sg=1)  : 중심 단어 -> 주변 단어 예측
# Ex > 'i love nlp'
# CBOW              : 'i', 'nlp'를 보고 'love' 예측
# Skip-gram         : 'love'를 보고 'i', 'nlp' 예측 (데이터가 적을 때 더 잘 작동)

# 단어 벡터 확인
vector = model.wv['love']               # 'love'라는 단어의 벡터 꺼낼 수 있음
print(vector)
print(vector.shape)

# 유사 단어 찾기
model.wv.most_similar('learning')       # 전체 단어 순위 보여줌
# 코사인 유사도로 계산 (-1 ~ 1사이 값) 

# 유사도 직접 계산
model.wv.similarity('fun', 'learning')  # 특정 두 단어 사이 유사도

[-8.7274825e-03  2.1301615e-03 -8.7354420e-04 -9.3190884e-03
 -9.4281426e-03 -1.4107180e-03  4.4324086e-03  3.7040710e-03
 -6.4986930e-03 -6.8730675e-03 -4.9994122e-03 -2.2868442e-03
 -7.2502876e-03 -9.6033178e-03 -2.7436293e-03 -8.3628409e-03
 -6.0388758e-03 -5.6709289e-03 -2.3441375e-03 -1.7069972e-03
 -8.9569986e-03 -7.3519943e-04  8.1525063e-03  7.6904297e-03
 -7.2061159e-03 -3.6668312e-03  3.1185520e-03 -9.5707225e-03
  1.4764392e-03  6.5244664e-03  5.7464195e-03 -8.7630618e-03
 -4.5171441e-03 -8.1401607e-03  4.5956374e-05  9.2636338e-03
  5.9733056e-03  5.0673080e-03  5.0610625e-03 -3.2429171e-03
  9.5521836e-03 -7.3564244e-03 -7.2703874e-03 -2.2653891e-03
 -7.7856064e-04 -3.2161034e-03 -5.9258583e-04  7.4888230e-03
 -6.9751858e-04 -1.6249407e-03  2.7443992e-03 -8.3591007e-03
  7.8558037e-03  8.5361041e-03 -9.5840869e-03  2.4462664e-03
  9.9049713e-03 -7.6658037e-03 -6.9669187e-03 -7.7365171e-03
  8.3959233e-03 -6.8133592e-04  9.1444086e-03 -8.1582209e-03
  3.7430846e-03  2.63504

np.float32(0.13724104)

In [ ]:
# 모델 저장 & 불러오기

# 모델 저장
model.save('customword2vec.model')                   # 파일로 저장

# 모델 불러오기
loaded_model = Word2Vec.load('customword2vec.model') # 파일에서 불러오기

In [11]:
# 한국어 Word2Vec
import pandas as pd
import re
from konlpy.tag import Okt

df = pd.read_csv('daum_movie_review.csv')
corpus = df['review'][:100]

In [ ]:
# 한글만 남기고 나머지 제거
sentences = [re.sub(r'[^가-힣\s]', '', text) for text in corpus]
okt = Okt()

# 한글 토큰화 (형태소 분석기 Okt 사용, 1글자가 넘는 단어만)
def korean_token(text):
    return [word for word, _ in okt.pos(text, stem=True) if len(word) > 1]

sentences = [korean_token(doc) for doc in sentences]

model = Word2Vec(
    sentences=sentences,    
    vector_size=1000,       # 영어 때는 100, 한국어는 1000
    window = 3,
    min_count = 5,          # 영어 때는 1, 한국어는 5
    sg = 1,
)

model.wv.most_similar('영화')

# 영어와 파라미터가 다름
# vector_size   : 100 -> 1000  한국어가 영어보다 어휘가 복잡함
# min_count     : 1 -> 5 

[('가다', 0.10001443326473236),
 ('있다', 0.08331053704023361),
 ('다음', 0.073957659304142),
 ('같다', 0.06599206477403641),
 ('모르다', 0.05515427514910698),
 ('자다', 0.054868899285793304),
 ('나오다', 0.05061757564544678),
 ('그래도', 0.04825276881456375),
 ('재밌다', 0.0442919097840786),
 ('정말', 0.025044571608304977)]

## 011.ipynb

In [ ]:
# Word2Vec는 미리 학습
# Embedding은 감성 분석 학습하면서 동시에 단어 벡터도 같이 업데이트
# => 감성 분석에 최적화된 단어 벡터가 만들어짐

In [14]:
# 데이터 로드 & 전처리

# 데이터 로드
import pandas as pd
import re
from konlpy.tag import Okt

df = pd.read_csv('daum_movie_review.csv')

# 컬럼 4개 
# review    : 리뷰 텍스트
# rating    : 평점 (1 ~ 10)
# date      : 날짜
# title     : 영화 제목

# 데이터 전처리
okt = Okt()

def clean_text(text):
    # 한글과 공백만 남기고 제거
    return re.sub(r'[^가-힣\s]', '', text)

def tokenize(text):
    # 형태소 분석 + 원형 복원 + 1글자 제거
    return [word for word , _ in okt.pos(text, stem=True) if len(word) > 1]

df['cleaned_review'] = df['review'].apply(clean_text)
df['data'] = df['cleaned_review'].apply(tokenize)


In [15]:
# 단어 사전 (Vocabulary) 만들기

from collections import Counter

# 모든 리뷰의 단어를 하나로 합치기
all_words = [ word for doc in df['data'] for word in doc ]

# 단어별 등장 횟수 세기
word_counts = Counter(all_words)

# 등장 횟수 2회 이상인 단어만 사전에 등록
# 특수 토큰 2개를 앞에 추가
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.items() if count >= 2]
# <PAD> : Padding Token
# 신경망은 입력 크기가 고정되야 하는데 문장마다 길이가 다름 (짧은 문장은 <PAD>로 채움)
# <UNK> : Unknown Token
# 학습 때 못 본 단어, 또는 등장 횟수가 적어서 사전에 없는 단어

# 단어 -> 인덱스 딕셔너리
word2idx = {word : idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)

In [18]:
# 문장을 숫자로 변환 & 패딩

def encode(tokens, max_len=50):
    # 단어를 인덱스로 변환, 사전에 없으면 <UNK>(1) 사용
    encoded = [word2idx.get(word, 1) for word in tokens]
    # dict.get(key, default)
    # word2idx['영화']          : 키가 없으면 -> KeyError 에러 발생
    # word2idx.get('영화', 1)   : 키가 없으면 -> 1 반환 (에러 없음)

    # max_len보다 길면 자르기
    encoded = encoded[:max_len]

    # max_len보다 짧으면 <PAD>(0)으로 채우기
    padded = encoded + [0] * (max_len - len(encoded))

    return padded

df['padded'] = df['data'].apply(encode)

In [19]:
# Dataset & DataLoader

import torch
from torch.utils.data import Dataset, DataLoader

class MovieReviewerDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.LongTensor(sequences)    # 패딩된 숫자 리스트 -> 텐서
        self.labels = torch.FloatTensor(labels)         # 정답 (0 or 1) -> 텐서

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]
    
# Pytorch에서 데이터를 다루는 기본 구조 (3개의 함수로 이뤄짐)
# __init__      : 데이터 저장 ( 인덱스 번호 => 정수(Long), 정답 레이블은 나중에 소수 계산 필요(Float) )
# __len__       : 데이터 개수 반환 => len(dataset)하면 몇 개의 리뷰가 있는지 반환
# __getitem__   : 인덱스로 데이터 하나 꺼내기

df['target'] = df['rating'].apply(lambda x : 0 if x > 5 else 1)
# rating > 5    => 0(긍정)
# rating <= 5   => 1(부정)

# 데이터 나누기
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    df['padded'].tolist(), df['target'].values, random_state=42, test_size=0.2
    )

# DataLoader : 데이터를 묶어서 모델에 넣어줌 -> 배치(Batch)
train_loader = DataLoader(MovieReviewerDataset(x_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(MovieReviewerDataset(x_test, y_test), batch_size=32, shuffle=False)

In [ ]:
# 모델 정의 (Embedding + MLP)
import torch.nn as nn

class EmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # 단어 인덱스를 벡터로 바꿔주는 레이어
        # vocab_size    : 사전에 있는 단어 수
        # embedding_dim : 각 단어를 몇 차원 벡터로 표현할지
        # padding_idx=0 : <PAD>(인덱스 0)는 벡터를 전부 0으로 -> 학습에서 제외 

        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),   # 차원 줄이기
            # 행렬 곱으로 차원을 변환
            nn.Dropout(0.5),                        # 과적합 방지 
            # (학습중 랜덤하게 50%)뉴런을 꺼서 과적합 방지
            nn.ReLU(),                              # 비선형 활성화
            nn.Linear(hidden_dim, 1)                # 최종 출력 (숫자 1개 -> 긍정 / 부정)
        )

    def forward(self, x):
        x = self.embedding(x)   # B L D : 배치, 문장 길이, 임배딩 차원
        x = x.mean(dim=1)       # B D   : 단어 벡터들의 평균
        return self.fc(x)

# 입력 x : (32, 50)             => 32개 리뷰, 각 50개 단어 인덱스
# embedding 후 : (32, 50, 128)  => 각 단어가 128차원 벡터로 임베딩
# mean 후 : (32, 128)           => 50개 단어 벡터를 평균 -> 리뷰 하나당 벡터 1개
# fc 후 : (32, 1)               => 리뷰 하나당 숫자 1개 출력

In [30]:
# 학습 (Training Loop)
import numpy as np
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# GPU가 있으면 'cuda' 없으면 'cpu'
from torch.optim import Adam

model = EmbeddingMLP(vocab_size, np.array(x_train).shape[1], 64).to(device)
criterion = nn.BCEWithLogitsLoss()  
# BCEWithLogitsLoss : 이진 분류 (긍정/부정)에 쓰는 손실 함수
# -> 모델 예측이 정답과 얼마나 다른지 계산
# 이 값이 작아지도록 학습

optimizer = Adam(model.parameters(), lr=1e-3)

epochs = 10
pbar = tqdm(range(epochs), desc="Training")

for epoch in pbar:
    model.train()
    local_loss = 0.0

    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)

        optimizer.zero_grad()               # 이전 gradient 초기화
        outputs = model(vecs).squeeze(1)    # 모델 예측
        loss = criterion(outputs, labels)   # 손실 계산
        loss.backward()                     # gradient 계산
        optimizer.step()                    # 파라미터 업데이트

        local_loss += loss.item()

    avg_loss = local_loss / len(train_loader)
    pbar.set_postfix(loss=f"{avg_loss:.4f}")

Training: 100%|██████████| 10/10 [00:05<00:00,  1.71it/s, loss=0.2123]


In [ ]:
# 모델 평가

model.eval()    # 학습 모드 -> 평가 모드로 전환 => Dropout이 꺼짐 (학습 때는 뉴런을 끄고, 평가 때는 전부 켜야 정확)
correct = 0.0
total = 0.0

with torch.no_grad():   # 평가 때는 파라미터 업데이트 필요 X
    for vecs, labels in test_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        preds = (torch.sigmoid(model(vecs)).squeeze(1) > 0.5).float()
        # model(vecs)        : 숫자 1개 출력 (범위 제한 X)
        # torch.sigmoid()    : 0 ~ 1 사이로 압축
        # > 0.5              : 0.5 넘으면 True(부정), 아니면 False(긍정)
        # squeeze(1)         : model(vecs)의 출력 (32, 1) => 32개 리뷰, 각각 숫자 1개   
        #                     labels의 shape (32,) 그냥 32개 => 쓸모없는 1차원 제거 (비교 가능하게 만들기)

        correct += (preds == labels).sum().item()
        total += labels.shape[0]

print(f"test accuracy : {(correct / total):.4f}")

test accuracy : 0.8424
